# 01c — Train: Direct Fine-tuning Baselines

Trains three conditions that test whether fine-tuning Flan-T5-XL on
**plan-level binary labels** (valid/invalid for the full plan) can approach
the performance of the stepwise entailment method (01a, FT_Combined).

## Why these conditions exist

The proposed method (FT_Combined) uses stepwise entailment labels — one label
per action step, identifying which precondition is violated. A natural question
is: can we achieve similar results by fine-tuning on the same task in the direct
format (full plan → valid/invalid)?

These three conditions answer that question at different data scales:

| Condition | Training data | Test set | Question answered |
|---|---|---|---|
| `FT50` | 50 plans (5 valid + 5 invalid per domain) | Full test set | Does any direct FT help at all? |
| `FT_Random` | ~1,014 plans (random 70% of test set) | Own 20% holdout | Does scaling up direct FT help? |
| `FT_Length` | ~1,015 plans (shortest 70% by action count) | Longest 20% | Does FT on easy plans generalise to hard plans? |

**Key finding (expected):** all three conditions collapse to predicting every
plan as invalid, regardless of training data scale. Binary plan-level labels
are too coarse to teach precondition checking — the model cannot learn which
action steps are executable from a single valid/invalid label per plan.

## Important note on comparability

`FT_Random` and `FT_Length` are evaluated on their own 20% holdout sets,
not the full test set, because 70% of the test set was used for training.
Their results are **not directly comparable** to conditions evaluated on the
full test set.

## Output

Adapters saved to:
- `paper2/adapters/FlanT5_Direct_FT50/`
- `paper2/adapters/FlanT5_Direct_FT_Random/`
- `paper2/adapters/FlanT5_Direct_FT_Length/`

## Prerequisites

Run `00_data_preparation.ipynb` first.

## 1. Setup

DEBUG mode runs a quick 4-record sanity check per condition.
Set `DEBUG=False` for full training runs.

`MAX_INPUT_LEN=2048` is critical for the direct format: logistics plans
can reach ~1,800 tokens. Using 1,024 would truncate ~30% of logistics records.

In [1]:
DEBUG   = False   # Set False for full training
DEBUG_N = 4      # records per condition in debug mode
print(f'Mode: {"DEBUG" if DEBUG else "FULL TRAINING"}')

Mode: FULL TRAINING


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip -q install -U transformers peft accelerate datasets scikit-learn bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 50.6 MB/s eta 0:00:00


In [4]:
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via HF_TOKEN secret')
except Exception:
    from huggingface_hub import login
    login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
import os, gc, json, re, math, random, shutil, warnings
import numpy as np
import torch
import torch.nn as nn
from collections import Counter, defaultdict
from dataclasses import dataclass as _dc

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
    TrainerCallback,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# -- Paths -------------------------------------------------------------------
ROOT        = '/content/drive/MyDrive/paper_codes/paper2'
DATA_DIR    = f'{ROOT}/data'
RESULTS_DIR = f'{ROOT}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

MANIFEST_PATH  = os.path.join(DATA_DIR, 'split_manifest.json')
DI_TEST_PATH   = os.path.join(DATA_DIR, 'planbench_direct_test.jsonl')

ADAPTER_FT50   = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT50',     'lora_adapter')
ADAPTER_RANDOM = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT_Random', 'lora_adapter')
ADAPTER_LENGTH = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT_Length', 'lora_adapter')
TOK_FT50       = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT50',     'tokenizer')
TOK_RANDOM     = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT_Random', 'tokenizer')
TOK_LENGTH     = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT_Length', 'tokenizer')
for d in [ADAPTER_FT50, ADAPTER_RANDOM, ADAPTER_LENGTH,
          TOK_FT50, TOK_RANDOM, TOK_LENGTH]:
    os.makedirs(d, exist_ok=True)

# -- Model config ------------------------------------------------------------
# MAX_INPUT_LEN=2048: direct prompts include the full plan text.
# Logistics domain plans can reach ~1800 tokens.
# Using 1024 would truncate ~30% of logistics records.
BASE_MODEL     = 'google/flan-t5-xl'
MAX_INPUT_LEN  = 2048
MAX_TARGET_LEN = 1

# -- Training config ---------------------------------------------------------
# BATCH_SIZE=1: direct prompts are long, larger batches cause OOM.
# GRAD_ACCUM=16: effective batch = 1 x 16 = 16.
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR         = 2e-4
MAX_EPOCHS = 50

# -- Split ratios for FT_Random and FT_Length --------------------------------
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
TEST_RATIO  = 0.20

DOMAINS = ['blocksworld', 'blocksworld_3', 'mystery', 'mystery_3', 'logistics']
DOMAIN_MAP = {
    'mystery_blocksworld'  : 'mystery',
    'mystery_blocksworld_3': 'mystery_3',
    'blocksworld'          : 'blocksworld',
    'blocksworld_3'        : 'blocksworld_3',
    'logistics'            : 'logistics',
}

use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
def cleanup(): gc.collect(); torch.cuda.empty_cache()

print(f'ROOT          : {ROOT}')
print(f'GPU           : {torch.cuda.get_device_name(0)}')
print(f'VRAM          : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'BF16          : {use_bf16}')
print(f'MAX_INPUT_LEN : {MAX_INPUT_LEN}')

ROOT          : /content/drive/MyDrive/paper_codes/paper2
GPU           : NVIDIA A100-SXM4-80GB
VRAM          : 85.1 GB
BF16          : True
MAX_INPUT_LEN : 2048


## 2. Mystery Abstraction and Direct Prompt Builder

### Mystery vocabulary abstraction
The Mystery and Mystery-3 domains use semantically meaningful terms that could
allow a model to exploit world knowledge. All such terms are replaced with
abstract tokens before prompt construction, matching the abstraction applied
in `00_data_preparation.ipynb` exactly.

### Direct prompt format
The direct prompt presents the full plan in a single query:
- Domain action rules
- Initial world state
- Goal state
- Full numbered action sequence

The model outputs a single token: `A` (plan valid) or `B` (plan invalid).

This is the structural opposite of the stepwise format: the model must
process the entire plan at once and produce a single verdict, without
identifying which step fails or checking preconditions individually.

In [6]:
# Mystery abstraction map - matches notebook 00 exactly
# Relations: Province->Rel1, Planet->Rel2, Harmony->Rel3, Pain->Rel4, Craves->Rel5
MYSTERY_ACTION_MAP = {
    'attack':'action1', 'succumb':'action2',
    'overcome':'action3', 'feast':'action4',
    'Attack':'Action1', 'Succumb':'Action2',
    'Overcome':'Action3', 'Feast':'Action4',
}
MYSTERY_REL_MAP = {
    'Province':'Rel1', 'province':'rel1',
    'Planet':'Rel2',   'planet':'rel2',
    'Harmony':'Rel3',  'harmony':'rel3',
    'Pain':'Rel4',     'pain':'rel4',
    'Craves':'Rel5',   'craves':'rel5',
}
MYSTERY_FULL_MAP = {**MYSTERY_ACTION_MAP, **MYSTERY_REL_MAP}

def abstract_mystery(text):
    for orig, repl in MYSTERY_FULL_MAP.items():
        text = re.sub(r'\b' + re.escape(orig) + r'\b', repl, text)
    return text

def build_direct_prompt(rules, init_facts, goal_facts, actions):
    # Builds the direct plan verification prompt.
    # Presents the full plan at once - all rules, initial state, goal, actions.
    # The model must determine from the entire sequence whether any action
    # violates its preconditions, without localising which step fails.
    rules_block = '\n'.join(f'- {r.strip()}' for r in rules if r.strip())
    init_block  = '\n'.join(f'- {f.strip()}' for f in init_facts if f.strip())
    goal_block  = '\n'.join(f'- {f.strip()}' for f in goal_facts if f.strip())
    acts_block  = '\n'.join(f'{i+1}. {a.strip()}' for i, a in enumerate(actions))
    return (
        'Task: Determine whether the following plan is valid '
        'given the rules and initial state.\n'
        f'Rules:\n{rules_block}\n\n'
        f'Initial state:\n{init_block}\n\n'
        f'Goal:\n{goal_block}\n\n'
        f'Plan:\n{acts_block}\n\n'
        'Output format: Answer A if the plan is valid, B if it is not.\n'
        'Answer:'
    )

assert abstract_mystery('Province object-a') == 'Rel1 object-a'
assert abstract_mystery('craves object-b')   == 'rel5 object-b'
print('Mystery abstraction: OK')
print('Direct prompt builder: OK')

Mystery abstraction: OK
Direct prompt builder: OK


## 3. Parsers

These functions re-parse raw HuggingFace PlanBench records to build direct
format prompts for the fewshot plans. The test-set direct records are loaded
from `planbench_direct_test.jsonl` (written by `00_data_preparation.ipynb`),
but fewshot plans must be rebuilt here because the direct test file only
contains test-set plans.

- `parse_rules` — extracts action precondition/effect rules from the query text
- `parse_query` — extracts initial state, goal text, and action sequence
- `parse_actions` — extracts the action list between `[PLAN]` and `[PLAN END]`
- `parse_ground_truth` — extracts validity label and failure mode
- `build_direct_record` — assembles a complete direct format record

Mystery domain text is abstracted before any parsing.

In [7]:
def parse_rules(query, domain):
    # Extract domain action rules from the query text.
    # Rules appear between the action description header and [STATEMENT].
    # Header lines like 'pick up a block' are filtered out.
    m = re.search(
        r'(?:I have the following restrictions on my actions:|'
        r'Here are the actions that can be performed:)(.*?)(?=\[STATEMENT\])',
        query, re.DOTALL | re.I)
    if not m: return []
    lines = [l.strip() for l in m.group(1).strip().split('\n') if l.strip()]
    lines = [l for l in lines if not re.match(r'for example[,.]', l, re.I)]
    skip = {
        'pick up a block', 'unstack a block from on top of another block',
        'put down a block', 'stack a block on top of another block',
        'attack object', 'feast object from another object',
        'succumb object', 'overcome object from another object',
    }
    lines = [l for l in lines if l.lower().rstrip('.') not in skip and len(l) > 10]
    if domain in ('mystery', 'mystery_3'):
        lines = [abstract_mystery(l) for l in lines]
    return lines

def parse_actions(text):
    # Extract action list from the plan section delimited by [PLAN]/[PLAN END].
    m = re.search(r'My plan is as follows:\s*(.*?)(?:\[PLAN END\]|$)',
                  text, re.DOTALL | re.I)
    if not m: return []
    lines = [re.sub(r'^\[?\d+\]?\.?\s*', '', l.strip())
             for l in m.group(1).strip().split('\n')]
    return [l for l in lines if l and not re.match(r'\[', l)]

def parse_ground_truth(gt):
    # Parse ground truth string into label and fail reason.
    # fail_reason: 'valid', 'action_fail', 'goal_fail', or 'missing'
    # Plans with missing ground truth are excluded from all datasets.
    if not gt: return {'label': 0, 'fail_reason': 'missing'}
    gt = str(gt).strip()
    if re.search(r'the above plan is valid', gt, re.I):
        return {'label': 1, 'fail_reason': 'valid'}
    if re.search(r'action at step (\d+)', gt, re.I):
        return {'label': 0, 'fail_reason': 'action_fail'}
    return {'label': 0, 'fail_reason': 'goal_fail'}

def parse_query(query, domain):
    # Extract structured fields from a PlanBench query string.
    # Finds the last [STATEMENT] block.
    # Returns None if the query is malformed or missing required sections.
    stmts = [m.start() for m in re.finditer(r'\[STATEMENT\]', query)]
    if not stmts: return None
    last = query[stmts[-1]:]
    im = re.search(r'As initial conditions I have that,?\s*(.*?)\s*My goal',
                   last, re.DOTALL | re.I)
    gm = re.search(r'My goal is to have that\s*(.*?)\s*My plan',
                   last, re.DOTALL | re.I)
    if not im or not gm: return None
    actions = parse_actions(last)
    if not actions: return None
    return {
        'rules'    : parse_rules(query, domain),
        'init_text': im.group(1).strip(),
        'goal_text': gm.group(1).strip(),
        'actions'  : actions,
    }

def parse_facts(text, domain):
    # Parse a facts text string into individual fact strings.
    # Splits on commas and 'and' conjunctions.
    if domain in ('mystery', 'mystery_3'):
        text = abstract_mystery(text)
    parts = re.split(r',|\band\b', text.lower().strip())
    return list({p.strip().strip('.') for p in parts if p.strip()})

def build_direct_record(plan_id, row, domain, split_label):
    # Build a single direct-format record from a raw HuggingFace row.
    # Returns None if the record cannot be parsed.
    query   = row.get('query', '')
    gt      = row.get('ground_truth_plan', '')
    parsed  = parse_query(query, domain)
    gt_info = parse_ground_truth(gt)
    if parsed is None or gt_info['fail_reason'] == 'missing':
        return None
    init_facts = parse_facts(parsed['init_text'], domain)
    goal_facts = parse_facts(parsed['goal_text'], domain)
    actions    = parsed['actions']
    if domain in ('mystery', 'mystery_3'):
        actions = [abstract_mystery(a) for a in actions]
    label_int = gt_info['label']
    prompt    = build_direct_prompt(parsed['rules'], init_facts, goal_facts, actions)
    return {
        'split'      : split_label,
        'domain'     : domain,
        'plan_id'    : plan_id,
        'label_int'  : label_int,
        'label'      : 'A' if label_int == 1 else 'B',
        'fail_reason': gt_info['fail_reason'],
        'n_actions'  : len(actions),
        'prompt'     : prompt,
    }

print('Parsers: OK')

Parsers: OK


## 4. Tokeniser and Dataset

Flan-T5-XL encoder-decoder tokeniser. Direct prompts are much longer than
stepwise prompts because they include the full plan text.

- `MAX_INPUT_LEN = 2048`: required for logistics plans (~1,800 tokens)
- `MAX_TARGET_LEN = 1`: target is a single token (`A` or `B`)
- `BATCH_SIZE = 1`: direct prompts are long; larger batches cause OOM
- `GRAD_ACCUM = 16`: effective batch size = 1 × 16 = 16

Label token IDs for `A` and `B` are extracted from the vocabulary for
the weighted loss function.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

tok_A = tokenizer('A', add_special_tokens=False).input_ids[0]
tok_B = tokenizer('B', add_special_tokens=False).input_ids[0]
print(f'Token IDs: A={tok_A}  B={tok_B}')

def tokenize_records(records):
    # Tokenize a list of direct records for seq2seq training.
    # Input: full prompt -> encoder input_ids + attention_mask
    # Target: 'A' or 'B' -> decoder labels
    enc = tokenizer(
        [r['prompt'] for r in records],
        max_length=MAX_INPUT_LEN, truncation=True,
        padding=False, return_tensors=None)
    lab = tokenizer(
        text_target=[r['label'] for r in records],
        max_length=MAX_TARGET_LEN + 1, truncation=True,
        padding=False, return_tensors=None)
    enc['labels'] = lab['input_ids']
    return enc

class PlanDataset(torch.utils.data.Dataset):
    # PyTorch dataset wrapping tokenized plan records.
    # Each item is a dict of tensors consumed by the HuggingFace Trainer.
    def __init__(self, records):
        self.data = tokenize_records(records)
        self.n    = len(records)
    def __len__(self): return self.n
    def __getitem__(self, i):
        return {k: torch.tensor(v[i]) for k, v in self.data.items()}

print('Tokenizer and PlanDataset: OK')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Token IDs: A=71  B=272
Tokenizer and PlanDataset: OK


## 5. Shared Training Components

### Weighted cross-entropy loss
Applied to all three conditions to address class imbalance in training data.
`w_c = N / (2 × N_c)` where N is total training records and N_c is count for
class c. This prevents the model from collapsing to predict the majority class.

### LoRA configuration
Same as 01a and 01b: r=8, alpha=16, dropout=0.05, target modules q/k/v/o.
Trainable parameters: ~9.4M / 2.86B (0.33%).

### Gradient checkpointing
Enabled for all conditions to fit Flan-T5-XL on a single GPU.

In [9]:
def get_class_weights(records):
    # Compute inverse-frequency class weights.
    n_A   = sum(1 for r in records if r['label'] == 'A')
    n_B   = sum(1 for r in records if r['label'] == 'B')
    total = n_A + n_B
    w_A   = total / (2 * n_A) if n_A > 0 else 1.0
    w_B   = total / (2 * n_B) if n_B > 0 else 1.0
    print(f'  Class weights: w_A={w_A:.3f} w_B={w_B:.3f}  '
          f'(n_A={n_A} n_B={n_B} total={total})')
    return w_A, w_B

def make_weighted_trainer(w_A, w_B):
    # Factory that returns a WeightedSeq2SeqTrainer class with baked-in weights.
    # Creates a fresh class each time to avoid device conflicts across conditions.
    class WeightedSeq2SeqTrainer(Seq2SeqTrainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels  = inputs.get('labels').clone()
            outputs = model(**inputs)
            logits  = outputs.logits
            weights = torch.ones(logits.size(-1), device=logits.device)
            weights[tok_A] = w_A
            weights[tok_B] = w_B
            loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
            # Shift for seq2seq: decoder input is offset by 1
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1))
            return (loss, outputs) if return_outputs else loss
    return WeightedSeq2SeqTrainer

def compute_metrics(eval_pred):
    # Compute accuracy and F1-macro for checkpoint selection.
    # Used with predict_with_generate=True so eval_pred contains
    # generated token sequences, not raw logits.
    pred_ids, label_ids = eval_pred
    pred_texts = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_ids  = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
    gold_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    def to_int(text):
        return 1 if text.strip().upper().startswith('A') else 0
    preds = [to_int(t) for t in pred_texts]
    golds = [to_int(t) for t in gold_texts]
    acc = accuracy_score(golds, preds)
    f1  = f1_score(golds, preds, average='macro', zero_division=0)
    return {'eval_accuracy': acc, 'eval_f1_macro': f1}

def build_lora_model():
    # Load Flan-T5-XL and apply LoRA. Called fresh for each training run
    # to ensure clean model state between conditions.
    cleanup()
    base = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16 if use_bf16 else torch.float32,
        device_map='auto',
    )
    base.gradient_checkpointing_enable()
    if hasattr(base, 'enable_input_require_grads'):
        base.enable_input_require_grads()
    model = get_peft_model(base, LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
        task_type=TaskType.SEQ_2_SEQ_LM,
        target_modules=['q', 'k', 'v', 'o'],
    ))
    if hasattr(model, 'generation_config'):
        model.generation_config.max_length = MAX_TARGET_LEN + 1
    return model

def make_training_args(output_dir, n_epochs, n_train, with_eval=True):
    # Build Seq2SeqTrainingArguments with dynamically scaled eval/warmup steps.
    # eval_steps fires approximately 5 times per training run.
    # predict_with_generate=True: consistent decoding between eval and inference.
    steps_per_epoch = max(1, math.ceil(n_train / (BATCH_SIZE * GRAD_ACCUM)))
    total_steps     = steps_per_epoch * (2 if DEBUG else n_epochs)
    eval_steps      = max(1, min(200, total_steps // 5))
    warmup_steps    = max(1, min(65, total_steps // 10))
    print(f'  steps/epoch={steps_per_epoch}  total={total_steps}  '
          f'eval_steps={eval_steps}  warmup={warmup_steps}')
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=2 if DEBUG else n_epochs,
        learning_rate=LR,
        warmup_steps=warmup_steps,
        lr_scheduler_type='cosine',
        weight_decay=0.0,
        eval_strategy='steps' if with_eval else 'no',
        eval_steps=5 if DEBUG else eval_steps,
        save_strategy='steps' if with_eval else 'no',
        save_steps=5 if DEBUG else eval_steps,
        save_total_limit=2,
        load_best_model_at_end=with_eval,
        metric_for_best_model='eval_f1_macro' if with_eval else None,
        greater_is_better=True if with_eval else None,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LEN + 1,
        logging_steps=1 if DEBUG else 50,
        report_to='none',
        fp16=False, bf16=bool(use_bf16),
        dataloader_num_workers=0,
        seed=SEED,
    )

print('Shared training components: OK')

Shared training components: OK


## 6. Load Manifest and Build FT50 Training Records

Loads the split manifest from `00_data_preparation.ipynb` to identify
the 5 `fewshot_invalid` and 5 `fewshot_valid` plans per domain.

**FT50 training set:** 10 plans per domain × 5 domains = 50 plans total
(5 valid + 5 invalid per domain, 25 A + 25 B labels).

Both `fewshot_invalid` and `fewshot_valid` plans are used here — this is the
only condition where `fewshot_valid` plans appear in training data.

Direct records are rebuilt from the raw HuggingFace dataset rather than
loaded from file, because `planbench_direct_test.jsonl` only contains
test-set plans.

In [10]:
manifest_raw = json.load(open(MANIFEST_PATH))
manifest     = manifest_raw['manifest']
print('Manifest loaded. Domains:', list(manifest.keys()))
print(f'  n_fewshot_invalid = {manifest_raw["n_fewshot_invalid_per_domain"]}/domain')
print(f'  n_fewshot_valid   = {manifest_raw["n_fewshot_valid_per_domain"]}/domain')

fewshot_invalid_ids = {}
fewshot_valid_ids   = {}
for domain, dom_map in manifest.items():
    fewshot_invalid_ids[domain] = sorted(
        [int(pid) for pid, lbl in dom_map.items() if lbl == 'fewshot_invalid'])
    fewshot_valid_ids[domain]   = sorted(
        [int(pid) for pid, lbl in dom_map.items() if lbl == 'fewshot_valid'])

for d in DOMAINS:
    print(f'  {d:15s}: inv={fewshot_invalid_ids.get(d,[])}  '
          f'val={fewshot_valid_ids.get(d,[])}' )

Manifest loaded. Domains: ['mystery', 'mystery_3', 'blocksworld', 'blocksworld_3', 'logistics']
  n_fewshot_invalid = 5/domain
  n_fewshot_valid   = 5/domain
  blocksworld    : inv=[299, 302, 326, 434, 464]  val=[12, 90, 105, 237, 372]
  blocksworld_3  : inv=[60, 64, 92, 143, 154]  val=[122, 153, 172, 183, 196]
  mystery        : inv=[299, 302, 326, 434, 464]  val=[12, 90, 105, 237, 372]
  mystery_3      : inv=[13, 24, 30, 32, 47]  val=[37, 54, 64, 77, 89]
  logistics      : inv=[3, 21, 96, 102, 146]  val=[48, 56, 140, 193, 244]


In [11]:
print('Loading PlanBench from HuggingFace...')
hf_ds = load_dataset('tasksource/planbench', 'task_3_plan_verification', split='train')
domain_groups = defaultdict(list)
for row in hf_ds:
    domain_groups[row['domain']].append(row)
print(f'Loaded {len(hf_ds)} rows')
for hf_d, rows in sorted(domain_groups.items()):
    print(f'  {hf_d}: {len(rows)} rows')

Loading PlanBench from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

task_3_plan_verification/train-00000-of-(…):   0%|          | 0.00/1.03M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1584 [00:00<?, ? examples/s]

Loaded 1584 rows
  blocksworld: 500 rows
  blocksworld_3: 199 rows
  logistics: 285 rows
  mystery_blocksworld: 500 rows
  mystery_blocksworld_3: 100 rows


In [12]:
# Build direct records for all fewshot plans (invalid + valid)
ft50_all = []

for hf_domain, internal_domain in DOMAIN_MAP.items():
    rows    = domain_groups.get(hf_domain, [])
    inv_ids = set(fewshot_invalid_ids.get(internal_domain, []))
    val_ids = set(fewshot_valid_ids.get(internal_domain, []))
    all_pids = inv_ids | val_ids

    built, skipped = 0, 0
    for plan_id, row in enumerate(rows):
        if plan_id not in all_pids: continue
        rec = build_direct_record(plan_id, row, internal_domain, 'fewshot')
        if rec is None: skipped += 1; continue
        ft50_all.append(rec)
        built += 1

    dom_cnt = Counter(r['label'] for r in ft50_all if r['domain']==internal_domain)
    print(f'  {internal_domain:15s}: {built} records '
          f'(A={dom_cnt["A"]} B={dom_cnt["B"]}) skipped={skipped}')

cnt_all = Counter(r['label'] for r in ft50_all)
print(f'\nFT50 total: {len(ft50_all)} records '
      f'(A={cnt_all["A"]} B={cnt_all["B"]})')
assert len(ft50_all) > 0, 'No FT50 records built!'


  mystery        : 10 records (A=5 B=5) skipped=0
  mystery_3      : 10 records (A=5 B=5) skipped=0
  blocksworld    : 10 records (A=5 B=5) skipped=0
  blocksworld_3  : 10 records (A=5 B=5) skipped=0
  logistics      : 10 records (A=5 B=5) skipped=0

FT50 total: 50 records (A=25 B=25)


## 7. Truncation Check

Verifies that `MAX_INPUT_LEN=2048` is sufficient for all FT50 training records.
Direct prompts include the full plan text — logistics plans with up to 47 actions
produce the longest prompts (~1,800 tokens).

Any truncated records would lose part of the plan text, potentially changing
the correct label. Truncation count is reported; if non-zero, `MAX_INPUT_LEN`
should be increased.

In [13]:
print('Checking token lengths for FT50 records...')
max_len_seen = 0
n_truncated  = 0
for r in ft50_all:
    toks = tokenizer(r['prompt'], truncation=False).input_ids
    if len(toks) > max_len_seen: max_len_seen = len(toks)
    if len(toks) > MAX_INPUT_LEN: n_truncated += 1
print(f'  max_tokens={max_len_seen}  truncated={n_truncated}/{len(ft50_all)}')
if n_truncated == 0:
    print(f'  MAX_INPUT_LEN={MAX_INPUT_LEN} is sufficient. No truncation.')
else:
    print(f'  WARNING: {n_truncated} records exceed MAX_INPUT_LEN={MAX_INPUT_LEN}')

Token indices sequence length is longer than the specified maximum sequence length for this model (518 > 512). Running this sequence through the model will result in indexing errors


Checking token lengths for FT50 records...
  max_tokens=1667  truncated=0/50
  MAX_INPUT_LEN=2048 is sufficient. No truncation.


## 8. FT50 Training (FlanT5_Direct_FT50)

Trains on all 50 fewshot plans for a fixed 50 epochs. No validation set,
no early stopping.

**Why fixed epochs?**
With only 50 plan-level records, a held-out validation set of 10 records
gives an extremely noisy F1-macro signal — unreliable for early stopping.
Fixed epochs are more robust:
- steps/epoch = ceil(50 / 16) = 4
- 50 epochs × 4 steps = 200 gradient updates

The FOLIO pre-training (from FT_Combined, 01a) is not used here — FT50
starts from the base Flan-T5-XL model, not the FOLIO-fine-tuned adapter.

**Expected result:** the model collapses to predicting every plan as invalid,
achieving F1-macro near 31% (the score of a constant invalid predictor on
the imbalanced test set). Binary plan-level labels cannot teach the model
which preconditions to check.

In [14]:
FT50_EPOCHS = 50   # fixed epoch count

if DEBUG:
    ft50_use = ft50_all[:DEBUG_N]
else:
    ft50_use = ft50_all

print(f'FT50 train: {len(ft50_use)} records (all 50 fewshot plans)')
w_A_ft50, w_B_ft50 = get_class_weights(ft50_use)
TrainerClass_ft50  = make_weighted_trainer(w_A_ft50, w_B_ft50)
train_ds_ft50      = PlanDataset(ft50_use)

FT50_CKPT = os.path.join(ROOT, 'checkpoints', 'FlanT5_Direct_FT50')
os.makedirs(FT50_CKPT, exist_ok=True)

steps_per_epoch_ft50 = max(1, math.ceil(len(train_ds_ft50) / (BATCH_SIZE * GRAD_ACCUM)))
total_steps_ft50     = steps_per_epoch_ft50 * (2 if DEBUG else FT50_EPOCHS)
warmup_ft50          = max(1, min(65, total_steps_ft50 // 10))
print(f'steps/epoch={steps_per_epoch_ft50}  '
      f'total_steps={total_steps_ft50}  warmup={warmup_ft50}')

model_ft50 = build_lora_model()
model_ft50.print_trainable_parameters()

args_ft50 = Seq2SeqTrainingArguments(
    output_dir=FT50_CKPT,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=2 if DEBUG else FT50_EPOCHS,
    learning_rate=LR,
    warmup_steps=warmup_ft50,
    lr_scheduler_type='cosine',
    weight_decay=0.0,
    eval_strategy='no',
    save_strategy='no',
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN + 1,
    logging_steps=1 if DEBUG else 20,
    report_to='none',
    fp16=False, bf16=bool(use_bf16),
    dataloader_num_workers=0, seed=SEED,
)

trainer_ft50 = TrainerClass_ft50(
    model=model_ft50, args=args_ft50,
    train_dataset=train_ds_ft50,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, model=model_ft50, padding=True, label_pad_token_id=-100),
)
trainer_ft50.train()

trainer_ft50.model.save_pretrained(ADAPTER_FT50)
tokenizer.save_pretrained(TOK_FT50)
print(f'\nAdapter   -> {ADAPTER_FT50}')
print(f'Tokenizer -> {TOK_FT50}')
print(f'Epochs    = {FT50_EPOCHS}')

del model_ft50, trainer_ft50
cleanup()


FT50 train: 50 records (all 50 fewshot plans)
  Class weights: w_A=1.000 w_B=1.000  (n_A=25 n_B=25 total=50)
steps/epoch=4  total_steps=200  warmup=20


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 9,437,184 || all params: 2,859,194,368 || trainable%: 0.3301


Step,Training Loss
20,132.850659
40,32.546341
60,1.814135
80,0.052248
100,0.012166
120,0.008729
140,0.008098
160,0.006213
180,0.005974
200,0.005911



Adapter   -> /content/drive/MyDrive/paper_codes/paper2/adapters/FlanT5_Direct_FT50/lora_adapter
Tokenizer -> /content/drive/MyDrive/paper_codes/paper2/adapters/FlanT5_Direct_FT50/tokenizer
Epochs    = 50


## 9. Build FT_Random and FT_Length Training Records

Both conditions use the full test set (excluding the original 50 fewshot plans),
split 70/10/20 by different strategies. Records are loaded from
`planbench_direct_test.jsonl` (written by `00_data_preparation.ipynb`).

**FT_Random (stratified random split):**
- Train: random 70% of test plans, stratified by domain and label
- Val: random 10%, used for early stopping
- Test: random 20% holdout
- Tests pure data scaling: does more direct fine-tuning data help?

**FT_Length (length-based split):**
- Train: shortest 70% of plans by action count
- Val: next 10% by length
- Test: longest 20% holdout
- Tests generalisation: does fine-tuning on easy (short) plans transfer to
  hard (long) plans?

Both conditions are evaluated on their own 20% holdout sets only.
Since 70% of the test set was used for training, evaluating on the full
test set would constitute train/test contamination.

Splits are saved to `paper2/data/direct_splits.json` for reproducibility.

In [15]:
# (Section 10 markdown above — FT_Random/FT_Length data loading follows in next cell)


## 9. (continued) Load and Split Test Records

Loads `planbench_direct_test.jsonl` and applies the 70/10/20 split strategies
defined above. Split indices are saved to `direct_splits.json`.

In [16]:
di_test_all  = [json.loads(l) for l in open(DI_TEST_PATH)]
di_by_domain = defaultdict(list)
for r in di_test_all:
    di_by_domain[r['domain']].append(r)
print(f'Test records loaded: {len(di_test_all)}')
for d in DOMAINS:
    cnt = Counter(r['label'] for r in di_by_domain[d])
    print(f'  {d:15s}: {len(di_by_domain[d])} (A={cnt["A"]} B={cnt["B"]})')

# -- RANDOM SPLIT ------------------------------------------------------------
random.seed(SEED)
rand_train, rand_val, rand_test = [], [], []

for d in DOMAINS:
    recs    = di_by_domain[d]
    valid_r = [r for r in recs if r['label_int'] == 1]
    inval_r = [r for r in recs if r['label_int'] == 0]
    for group in [valid_r, inval_r]:
        random.shuffle(group)
        n        = len(group)
        n_test   = max(1, round(n * TEST_RATIO))
        n_val    = max(1, round(n * VAL_RATIO))
        n_train  = n - n_test - n_val
        rand_train.extend(group[:n_train])
        rand_val.extend(group[n_train:n_train + n_val])
        rand_test.extend(group[n_train + n_val:])

print(f'\nRandom split:')
print(f'  Train: {len(rand_train)}  '
      f'(A={sum(1 for r in rand_train if r["label"]=="A")} '
      f'B={sum(1 for r in rand_train if r["label"]=="B")})')
print(f'  Val  : {len(rand_val)}')
print(f'  Test : {len(rand_test)}  '
      f'(A={sum(1 for r in rand_test if r["label"]=="A")} '
      f'B={sum(1 for r in rand_test if r["label"]=="B")})')

# -- LENGTH SPLIT ------------------------------------------------------------
random.seed(SEED)
len_train, len_val, len_test = [], [], []
length_thresholds = {}

for d in DOMAINS:
    recs   = sorted(di_by_domain[d], key=lambda x: x['n_actions'])
    n      = len(recs)
    n_test = max(1, round(n * TEST_RATIO))
    pool      = recs[:n - n_test]    # shortest 80%
    test_set  = recs[n - n_test:]    # longest 20%
    length_thresholds[d] = {
        'pool_max': pool[-1]['n_actions']    if pool     else 0,
        'test_min': test_set[0]['n_actions'] if test_set else 0,
    }
    random.shuffle(pool)
    n_val_pool = max(1, round(len(pool) * (VAL_RATIO / (TRAIN_RATIO + VAL_RATIO))))
    len_val.extend(pool[:n_val_pool])
    len_train.extend(pool[n_val_pool:])
    len_test.extend(test_set)

print(f'\nLength split:')
for d in DOMAINS:
    t = length_thresholds[d]
    print(f'  {d:15s}: train/val <= {t["pool_max"]} actions  '
          f'test >= {t["test_min"]} actions')
print(f'  Train: {len(len_train)} (avg {np.mean([r["n_actions"] for r in len_train]):.1f} actions)')
print(f'  Val  : {len(len_val)}')
print(f'  Test : {len(len_test)} (avg {np.mean([r["n_actions"] for r in len_test]):.1f} actions)')

# Save splits for reproducibility
splits_log = {
    'seed': SEED,
    'random': {
        'train_plan_ids': {d: [r['plan_id'] for r in rand_train if r['domain']==d]
                           for d in DOMAINS},
        'val_plan_ids':   {d: [r['plan_id'] for r in rand_val   if r['domain']==d]
                           for d in DOMAINS},
        'test_plan_ids':  {d: [r['plan_id'] for r in rand_test  if r['domain']==d]
                           for d in DOMAINS},
    },
    'length': {
        'thresholds'    : length_thresholds,
        'train_plan_ids': {d: [r['plan_id'] for r in len_train if r['domain']==d]
                           for d in DOMAINS},
        'val_plan_ids':   {d: [r['plan_id'] for r in len_val   if r['domain']==d]
                           for d in DOMAINS},
        'test_plan_ids':  {d: [r['plan_id'] for r in len_test  if r['domain']==d]
                           for d in DOMAINS},
    },
}
splits_path = os.path.join(DATA_DIR, 'direct_splits.json')
with open(splits_path, 'w') as f:
    json.dump(splits_log, f, indent=2)
print(f'\nSplits saved -> {splits_path}')

Test records loaded: 1451
  blocksworld    : 474 (A=319 B=155)
  blocksworld_3  : 139 (A=50 B=89)
  mystery        : 474 (A=319 B=155)
  mystery_3      : 89 (A=50 B=39)
  logistics      : 275 (A=93 B=182)

Random split:
  Train: 1014  (A=581 B=433)
  Val  : 146
  Test : 291  (A=167 B=124)

Length split:
  blocksworld    : train/val <= 10 actions  test >= 10 actions
  blocksworld_3  : train/val <= 6 actions  test >= 6 actions
  mystery        : train/val <= 10 actions  test >= 10 actions
  mystery_3      : train/val <= 6 actions  test >= 6 actions
  logistics      : train/val <= 34 actions  test >= 34 actions
  Train: 1015 (avg 8.4 actions)
  Val  : 145
  Test : 291 (avg 15.2 actions)

Splits saved -> /content/drive/MyDrive/paper_codes/paper2/data/direct_splits.json


## 10. FT_Random Training (FlanT5_Direct_FT_Random)

Trains on the random 70% split (~1,014 plans). Early stopping on 10% val
F1-macro, patience=3. Max 50 epochs.

**Expected result:** same collapse as FT50 — predicts every plan as invalid.
Even with ~20× more training data than FT50, binary plan-level labels cannot
teach precondition checking. The supervision signal is too coarse.

In [17]:
if DEBUG:
    rand_train_use = rand_train[:DEBUG_N]
    rand_val_use   = rand_val[:max(1, DEBUG_N//2)]
else:
    rand_train_use = rand_train
    rand_val_use   = rand_val

print(f'FlanT5_Direct_FT_Random:')
print(f'  Train: {len(rand_train_use)}  Val: {len(rand_val_use)}')
w_A_r, w_B_r  = get_class_weights(rand_train_use)
TrainerClass_r = make_weighted_trainer(w_A_r, w_B_r)
train_ds_r     = PlanDataset(rand_train_use)
val_ds_r       = PlanDataset(rand_val_use)
RAND_CKPT      = os.path.join(ROOT, 'checkpoints', 'FlanT5_Direct_FT_Random')
os.makedirs(RAND_CKPT, exist_ok=True)

model_r = build_lora_model()
model_r.print_trainable_parameters()
args_r  = make_training_args(RAND_CKPT, MAX_EPOCHS, len(train_ds_r))

trainer_r = TrainerClass_r(
    model=model_r, args=args_r,
    train_dataset=train_ds_r,
    eval_dataset=val_ds_r,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, model=model_r, padding=True, label_pad_token_id=-100),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)
trainer_r.train()
best_r = trainer_r.state.best_metric
print(f'Best val F1-macro: {best_r:.4f}' if best_r is not None else 'Best: N/A')

trainer_r.model.save_pretrained(ADAPTER_RANDOM)
tokenizer.save_pretrained(TOK_RANDOM)
print(f'Adapter -> {ADAPTER_RANDOM}')
del model_r, trainer_r
cleanup()

FlanT5_Direct_FT_Random:
  Train: 1014  Val: 146
  Class weights: w_A=0.873 w_B=1.171  (n_A=581 n_B=433 total=1014)


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 9,437,184 || all params: 2,859,194,368 || trainable%: 0.3301
  steps/epoch=64  total=3200  eval_steps=200  warmup=65


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
200,0.001912,10.082192,0.431507,0.301435
400,0.000366,11.199914,0.431507,0.301435
600,0.000160,11.717037,0.431507,0.301435
800,0.007503,8.889771,0.431507,0.301435
1000,0.000266,11.183219,0.431507,0.301435
1200,0.000113,11.910531,0.431507,0.301435


Best val F1-macro: 0.3014
Adapter -> /content/drive/MyDrive/paper_codes/paper2/adapters/FlanT5_Direct_FT_Random/lora_adapter


## 11. FT_Length Training (FlanT5_Direct_FT_Length)

Trains on the shortest 70% of plans by action count (~1,015 plans).
Evaluates on the longest 20% holdout (~146 plans).
Early stopping on 10% val F1-macro, patience=3. Max 50 epochs.

**Expected result:** same collapse as FT_Random. Additionally tests
whether there is a length-based generalisation gap — whether difficulty
scales with plan length. In practice, the model cannot generalise even
within the same length range.

## 12. Summary

All three direct fine-tuning conditions are complete. Results on PlanBench
are reported in `02_eval_flant5.ipynb`.

**Key expected finding across all three conditions:**
The model predicts every plan as invalid, regardless of training data scale
(50 plans vs ~1,000 plans) or split strategy (random vs length-based).
This confirms that binary plan-level labels provide insufficient supervision
for learning precondition checking — a single valid/invalid label per plan
carries no information about which action steps are executable or why a plan
fails.

This motivates the stepwise entailment approach: by providing one label per
action step, the model receives a direct learning signal for the underlying
logical reasoning task.

In [20]:
print('=== 01c TRAINING COMPLETE ===')
print()
print('FlanT5_Direct_FT50')
print(f'  Training data : 5 fewshot_invalid + 5 fewshot_valid per domain (50 total)')
print(f'  Fixed epochs  : 50')
print(f'  Adapter       : {ADAPTER_FT50}')
print()
print('FlanT5_Direct_FT_Random')
print(f'  Training data : random 70% of test set ({len(rand_train)} plans)')
print(f'  Test set      : random 20% holdout ({len(rand_test)} plans)')
print(f'  Adapter       : {ADAPTER_RANDOM}')
print()
print()
print('Splits saved   :', os.path.join(DATA_DIR, 'direct_splits.json'))
print()
print('Next step: run 02_eval_flant5.ipynb')


=== 01c TRAINING COMPLETE ===

FlanT5_Direct_FT50
  Training data : 5 fewshot_invalid + 5 fewshot_valid per domain (50 total)
  Fixed epochs  : 50
  Adapter       : /content/drive/MyDrive/paper_codes/paper2/adapters/FlanT5_Direct_FT50/lora_adapter

FlanT5_Direct_FT_Random
  Training data : random 70% of test set (1014 plans)
  Test set      : random 20% holdout (291 plans)
  Adapter       : /content/drive/MyDrive/paper_codes/paper2/adapters/FlanT5_Direct_FT_Random/lora_adapter


Splits saved   : /content/drive/MyDrive/paper_codes/paper2/data/direct_splits.json

Next step: run 02_eval_flant5.ipynb
